In [4]:
import numpy as np

# p_s_0[s]
p_s_0 = np.array([0.25, 0.25, 0.25, 0.25])

# p_a[a]
p_a = np.array([0.25, 0.25, 0.25, 0.25])

# p_o_given_s[o, s]
p_o_given_s = np.array([[0.868, 0.064, 0.064, 0.005],
                        [0.064, 0.868, 0.005, 0.064],
                        [0.064, 0.005, 0.868, 0.064],
                        [0.005, 0.064, 0.064, 0.868]])

# p_trans[s_t, a_t, s_t+1]
p_trans = np.zeros((4, 4, 4))
# s_t = 0 (A: top left)
p_trans[0, :, :] = np.array([
    [0.877, 0.120, 0.003, 0.000],  # up
    [0.106, 0.774, 0.014, 0.106],  # right
    [0.106, 0.014, 0.774, 0.106],  # down
    [0.877, 0.003, 0.120, 0.000],  # left
])
# s_t = 1 (B: top right)
p_trans[1, :, :] = np.array([
    [0.120, 0.877, 0.000, 0.003],  # up
    [0.003, 0.877, 0.000, 0.120],  # right
    [0.014, 0.106, 0.106, 0.774],  # down
    [0.774, 0.106, 0.106, 0.014],  # left
])
# s_t = 2 (C: bottom left)
p_trans[2, :, :] = np.array([
    [0.774, 0.106, 0.106, 0.014],  # up
    [0.014, 0.106, 0.106, 0.774],  # right
    [0.003, 0.000, 0.877, 0.120],  # down
    [0.120, 0.000, 0.877, 0.003],  # left
])
# s_t = 3 (D: bottom right)
p_trans[3, :, :] = np.array([
    [0.106, 0.774, 0.014, 0.106],  # up
    [0.000, 0.120, 0.003, 0.877],  # right
    [0.000, 0.003, 0.120, 0.877],  # down
    [0.106, 0.014, 0.774, 0.106],  # left
])

In [5]:
p_o0_o1_o2_s0_s1_s2_a0_a1 = np.einsum(
    'i, ai, j, ijk, bk, l, klm, cm -> abcikmjl',
    p_s_0,         # s_0         -> i
    p_o_given_s,   # o_0, s_0    -> a, i
    p_a,           # a_0         -> j
    p_trans,       # s_0, a_0, s_1 -> i, j, k
    p_o_given_s,   # o_1, s_1    -> b, k
    p_a,           # a_1         -> l
    p_trans,       # s_1, a_1, s_2 -> k, l, m
    p_o_given_s,   # o_2, s_2    -> c, m
)

In [20]:
p_o0_cond = np.einsum('abikmjl -> a', p_o0_o1_o2_s0_s1_s2_a0_a1[:, :, 2, :, :, :, :, :])
p_o0_cond /= p_o0_cond.sum()
p_o1_cond = np.einsum('abikmjl -> b', p_o0_o1_o2_s0_s1_s2_a0_a1[:, :, 2, :, :, :, :, :])
p_o1_cond /= p_o1_cond.sum()
p_o2_cond = [0.0, 0.0, 1.0, 0.0]
p_s0_cond = np.einsum('abikmjl -> i', p_o0_o1_o2_s0_s1_s2_a0_a1[:, :, 2, :, :, :, :, :])
p_s0_cond /= p_s0_cond.sum()
p_s1_cond = np.einsum('abikmjl -> k', p_o0_o1_o2_s0_s1_s2_a0_a1[:, :, 2, :, :, :, :, :])
p_s1_cond /= p_s1_cond.sum()
p_s2_cond = np.einsum('abikmjl -> m', p_o0_o1_o2_s0_s1_s2_a0_a1[:, :, 2, :, :, :, :, :])
p_s2_cond /= p_s2_cond.sum()
p_a0_cond = np.einsum('abikmjl -> j', p_o0_o1_o2_s0_s1_s2_a0_a1[:, :, 2, :, :, :, :, :])
p_a0_cond /= p_a0_cond.sum()
p_a1_cond = np.einsum('abikmjl -> l', p_o0_o1_o2_s0_s1_s2_a0_a1[:, :, 2, :, :, :, :, :])
p_a1_cond /= p_a1_cond.sum()

print('A', 'B', 'C', 'D')
print('o0', p_o0_cond)
print('o1', p_o1_cond)
print('o2', p_o2_cond)
print('s0', p_s0_cond)
print('s1', p_s1_cond)
print('s2', p_s2_cond)
print('UP', 'RIGHT', 'DOWN', 'LEFT')
print('a0', p_a0_cond)
print('a1', p_a1_cond)

A B C D
o0 [0.24890311 0.17963696 0.32255683 0.24890311]
o1 [0.23767536 0.09936013 0.42528916 0.23767536]
o2 [0.0, 0.0, 1.0, 0.0]
s0 [0.24852619 0.16858691 0.33436072 0.24852619]
s1 [0.23344031 0.07753596 0.45558342 0.23344031]
s2 [0.06393606 0.004995   0.86713287 0.06393606]
UP RIGHT DOWN LEFT
a0 [0.1671131 0.1671131 0.3328869 0.3328869]
a1 [0.06097627 0.06097627 0.43902373 0.43902373]


In [21]:
p_o0_cond = [0.0, 1.0, 0.0, 0.0]
p_o1_cond = np.einsum('bikmjl -> b', p_o0_o1_o2_s0_s1_s2_a0_a1[1, :, 2, :, :, :, :, :])
p_o1_cond /= p_o1_cond.sum()
p_o2_cond = [0.0, 0.0, 1.0, 0.0]
p_s0_cond = np.einsum('bikmjl -> i', p_o0_o1_o2_s0_s1_s2_a0_a1[1, :, 2, :, :, :, :, :])
p_s0_cond /= p_s0_cond.sum()
p_s1_cond = np.einsum('bikmjl -> k', p_o0_o1_o2_s0_s1_s2_a0_a1[1, :, 2, :, :, :, :, :])
p_s1_cond /= p_s1_cond.sum()
p_s2_cond = np.einsum('bikmjl -> m', p_o0_o1_o2_s0_s1_s2_a0_a1[1, :, 2, :, :, :, :, :])
p_s2_cond /= p_s2_cond.sum()
p_a0_cond = np.einsum('bikmjl -> j', p_o0_o1_o2_s0_s1_s2_a0_a1[1, :, 2, :, :, :, :, :])
p_a0_cond /= p_a0_cond.sum()
p_a1_cond = np.einsum('bikmjl -> l', p_o0_o1_o2_s0_s1_s2_a0_a1[1, :, 2, :, :, :, :, :])
p_a1_cond /= p_a1_cond.sum()

print('A', 'B', 'C', 'D')
print('o0', p_o0_cond)
print('o1', p_o1_cond)
print('o2', p_o2_cond)
print('s0', p_s0_cond)
print('s1', p_s1_cond)
print('s2', p_s2_cond)
print('UP', 'RIGHT', 'DOWN', 'LEFT')
print('a0', p_a0_cond)
print('a1', p_a1_cond)

A B C D
o0 [0.0, 1.0, 0.0, 0.0]
o1 [0.28971229 0.21028771 0.21028771 0.28971229]
o2 [0.0, 0.0, 1.0, 0.0]
s0 [0.088455   0.81379274 0.00929727 0.088455  ]
s1 [0.30335839 0.19664161 0.19664161 0.30335839]
s2 [0.088455   0.00929727 0.81379274 0.088455  ]
UP RIGHT DOWN LEFT
a0 [0.14785944 0.14785944 0.35214056 0.35214056]
a1 [0.0520355 0.0520355 0.4479645 0.4479645]


In [22]:
p_o0_cond = [0.0, 1.0, 0.0, 0.0]
p_o1_cond = [1.0, 0.0, 0.0, 0.0]
p_o2_cond = [0.0, 0.0, 1.0, 0.0]
p_s0_cond = np.einsum('ikmjl -> i', p_o0_o1_o2_s0_s1_s2_a0_a1[1, 0, 2, :, :, :, :, :])
p_s0_cond /= p_s0_cond.sum()
p_s1_cond = np.einsum('ikmjl -> k', p_o0_o1_o2_s0_s1_s2_a0_a1[1, 0, 2, :, :, :, :, :])
p_s1_cond /= p_s1_cond.sum()
p_s2_cond = np.einsum('ikmjl -> m', p_o0_o1_o2_s0_s1_s2_a0_a1[1, 0, 2, :, :, :, :, :])
p_s2_cond /= p_s2_cond.sum()
p_a0_cond = np.einsum('ikmjl -> j', p_o0_o1_o2_s0_s1_s2_a0_a1[1, 0, 2, :, :, :, :, :])
p_a0_cond /= p_a0_cond.sum()
p_a1_cond = np.einsum('ikmjl -> l', p_o0_o1_o2_s0_s1_s2_a0_a1[1, 0, 2, :, :, :, :, :])
p_a1_cond /= p_a1_cond.sum()

print('A', 'B', 'C', 'D')
print('o0', p_o0_cond)
print('o1', p_o1_cond)
print('o2', p_o2_cond)
print('s0', p_s0_cond)
print('s1', p_s1_cond)
print('s2', p_s2_cond)
print('UP', 'RIGHT', 'DOWN', 'LEFT')
print('a0', p_a0_cond)
print('a1', p_a1_cond)

A B C D
o0 [0.0, 1.0, 0.0, 0.0]
o1 [1.0, 0.0, 0.0, 0.0]
o2 [0.0, 0.0, 1.0, 0.0]
s0 [0.13184025 0.83888893 0.00584958 0.02342124]
s1 [0.90797678 0.04339647 0.04339647 0.00523028]
s2 [0.13184025 0.00584958 0.83888893 0.02342124]
UP RIGHT DOWN LEFT
a0 [0.18581822 0.02985385 0.04621086 0.73811707]
a1 [0.06291615 0.03419075 0.70505477 0.19783833]
